# Actividad 3: Detección de Objetos con Visión Artificial
**Estudiante:** Stephanie Mercedes Machado Alvarado — Ingeniería en Sistemas, UNAH Campus Comayagua

**URL de la aplicación:** _(pegar aquí la URL de Streamlit)_

## Objetivo
Implementar un modelo de detección de objetos con visión artificial, documentar el proceso experimental y desplegar una aplicación web pública en Streamlit Community Cloud.

## 1. Marco teórico
- **Clasificación** dice *qué* hay en una imagen; **detección** dice *qué* hay y *dónde* (cajas delimitadoras + clase + confianza).
- **YOLO (You Only Look Once)** es un detector de una sola etapa: procesa la imagen completa en una pasada de la red, por eso es rápido y sirve en tiempo real.
- **YOLOv8** (Ultralytics) está preentrenado con **COCO** (80 clases: persona, carro, perro, silla, celular, etc.).
- **Confianza (conf):** probabilidad mínima para aceptar una detección.
- **IoU y NMS:** *Intersection over Union* mide cuánto se solapan dos cajas; *Non-Maximum Suppression* elimina cajas duplicadas que superan el umbral IoU.

In [ ]:
# Paso 1: Instalar librerías
!pip install -q ultralytics streamlit

In [ ]:
# Paso 2: Importar y cargar modelos
import time, urllib.request
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

modelo_n = YOLO('yolov8n.pt')   # nano: 3.2 M parámetros
modelo_s = YOLO('yolov8s.pt')   # small: 11.2 M parámetros
print('Clases disponibles:', len(modelo_n.names))
print(list(modelo_n.names.values()))

In [ ]:
# Paso 3: Imágenes de prueba
urls = {
    'bus.jpg': 'https://ultralytics.com/images/bus.jpg',
    'zidane.jpg': 'https://ultralytics.com/images/zidane.jpg',
}
for nombre, url in urls.items():
    urllib.request.urlretrieve(url, nombre)

# Opcional: subir tus propias fotos
# from google.colab import files
# subidas = files.upload()
imagenes = list(urls.keys())

In [ ]:
# Paso 4: Primera detección
def mostrar(resultado, titulo=''):
    plt.figure(figsize=(9, 7))
    plt.imshow(resultado.plot()[:, :, ::-1])
    plt.title(titulo); plt.axis('off'); plt.show()

for img in imagenes:
    r = modelo_n.predict(img, conf=0.25, verbose=False)[0]
    mostrar(r, f'{img} — {len(r.boxes)} objetos')
    for c, p in zip(r.boxes.cls, r.boxes.conf):
        print(f'  {modelo_n.names[int(c)]:<12} {float(p):.2f}')

## 2. Experimento 1: efecto del umbral de confianza
Se procesa la misma imagen con distintos umbrales para ver cómo cambia el número de detecciones.

In [ ]:
umbrales = [0.1, 0.25, 0.5, 0.75]
fig, ejes = plt.subplots(1, len(umbrales), figsize=(20, 6))
filas = []
for ax, u in zip(ejes, umbrales):
    r = modelo_n.predict('bus.jpg', conf=u, verbose=False)[0]
    ax.imshow(r.plot()[:, :, ::-1]); ax.set_title(f'conf={u} → {len(r.boxes)} objetos'); ax.axis('off')
    filas.append({'Confianza': u, 'Detecciones': len(r.boxes)})
plt.show()
pd.DataFrame(filas)

**Observación:** con umbral bajo aparecen más objetos pero también falsos positivos; con umbral alto quedan solo las detecciones seguras y se pierden objetos pequeños o parcialmente visibles. Un valor de **0.25–0.5** es un buen equilibrio.

## 3. Experimento 2: YOLOv8n vs YOLOv8s
Se compara velocidad y número de detecciones de ambos modelos.

In [ ]:
resultados = []
for nombre, m in [('YOLOv8n', modelo_n), ('YOLOv8s', modelo_s)]:
    m.predict('bus.jpg', verbose=False)  # calentamiento
    for img in imagenes:
        t0 = time.time()
        r = m.predict(img, conf=0.25, verbose=False)[0]
        ms = (time.time() - t0) * 1000
        conf_media = float(r.boxes.conf.mean()) if len(r.boxes) else 0
        resultados.append({'Modelo': nombre, 'Imagen': img, 'Detecciones': len(r.boxes),
                           'Confianza media': round(conf_media, 3), 'Tiempo (ms)': round(ms, 1)})
df = pd.DataFrame(resultados)
df

In [ ]:
resumen = df.groupby('Modelo')[['Detecciones', 'Confianza media', 'Tiempo (ms)']].mean()
resumen.plot(kind='bar', subplots=True, layout=(1, 3), figsize=(15, 4), legend=False, rot=0)
plt.tight_layout(); plt.show()
resumen

**Observación:** YOLOv8s suele dar mayor confianza media y detectar más objetos, pero tarda más. Para una app web en servidor gratuito (CPU), **YOLOv8n** es la opción por defecto y YOLOv8s queda como alternativa en la app.

## 4. Experimento 3: filtrar clases
Solo se buscan las clases de interés, por ejemplo personas y vehículos.

In [ ]:
ids = [k for k, v in modelo_n.names.items() if v in ['person', 'bus', 'car']]
r = modelo_n.predict('bus.jpg', conf=0.25, classes=ids, verbose=False)[0]
mostrar(r, 'Solo person / bus / car')

## 5. Aplicación web en Streamlit
Las siguientes celdas generan los archivos que se suben a GitHub.

In [ ]:
%%writefile app.py
import time
import numpy as np
import pandas as pd
import streamlit as st
from PIL import Image
from ultralytics import YOLO

st.set_page_config(page_title="Detección de objetos YOLOv8", page_icon="🔎", layout="wide")
st.title("🔎 Detección de objetos con YOLOv8")
st.write("Sube una imagen o toma una foto. El modelo YOLOv8 (entrenado con COCO, 80 clases) "
         "detecta personas, vehículos, animales, objetos de uso diario y más.")

@st.cache_resource
def cargar_modelo(nombre):
    return YOLO(nombre)

# --- Barra lateral ---
st.sidebar.header("⚙️ Parámetros")
nombre_modelo = st.sidebar.selectbox("Modelo", ["yolov8n.pt", "yolov8s.pt"],
                                     help="n = más rápido, s = más preciso")
conf = st.sidebar.slider("Confianza mínima", 0.05, 0.95, 0.25, 0.05)
iou = st.sidebar.slider("Umbral IoU (NMS)", 0.1, 0.9, 0.45, 0.05)
modelo = cargar_modelo(nombre_modelo)
clases = st.sidebar.multiselect("Filtrar clases (vacío = todas)", list(modelo.names.values()))

fuente = st.radio("Fuente", ["📁 Subir imagen", "📷 Cámara"], horizontal=True)
archivo = st.file_uploader("Imagen", type=["jpg", "jpeg", "png"]) if fuente == "📁 Subir imagen" \
          else st.camera_input("Toma una foto")

if archivo is not None:
    img = Image.open(archivo).convert("RGB")
    ids = [k for k, v in modelo.names.items() if v in clases] or None

    t0 = time.time()
    res = modelo.predict(np.array(img), conf=conf, iou=iou, classes=ids, verbose=False)[0]
    ms = (time.time() - t0) * 1000

    col1, col2 = st.columns(2)
    col1.subheader("Original")
    col1.image(img, use_container_width=True)
    col2.subheader("Detecciones")
    col2.image(res.plot()[:, :, ::-1], use_container_width=True)

    n = len(res.boxes)
    c1, c2 = st.columns(2)
    c1.metric("Objetos detectados", n)
    c2.metric("Tiempo de inferencia", f"{ms:.0f} ms")

    if n:
        df = pd.DataFrame({
            "Clase": [modelo.names[int(c)] for c in res.boxes.cls],
            "Confianza": res.boxes.conf.numpy().round(2),
            "Caja (x1, y1, x2, y2)": [tuple(map(int, b)) for b in res.boxes.xyxy.numpy()],
        })
        st.subheader("Detalle")
        st.dataframe(df, use_container_width=True)
        st.subheader("Conteo por clase")
        st.bar_chart(df["Clase"].value_counts())
    else:
        st.info("No se detectaron objetos. Prueba a bajar la confianza mínima.")


In [ ]:
%%writefile requirements.txt
streamlit
ultralytics
opencv-python-headless
Pillow
pandas


In [ ]:
%%writefile packages.txt
libgl1
libglib2.0-0


In [ ]:
# Descargar los 3 archivos para subirlos a GitHub
from google.colab import files
for f in ['app.py', 'requirements.txt', 'packages.txt']:
    files.download(f)

## 6. Despliegue en Streamlit Community Cloud
1. Crear un repositorio **público** en GitHub y subir `app.py`, `requirements.txt` y `packages.txt`.
2. Entrar a https://share.streamlit.io con GitHub → **Create app**.
3. Seleccionar el repositorio, rama `main`, archivo `app.py` → **Deploy**.
4. Copiar la URL (`https://<nombre>.streamlit.app`) y pegarla al inicio de este cuaderno y en la entrega.

**Funciones de la app:** subir imagen o usar la cámara, elegir modelo (n/s), ajustar confianza e IoU, filtrar clases, ver tabla de detecciones, conteo por clase y tiempo de inferencia.

## 7. Conclusiones
- YOLOv8 permite detectar 80 clases de objetos con un modelo preentrenado, sin necesidad de entrenar desde cero.
- El umbral de confianza controla el equilibrio entre detectar más objetos y evitar falsos positivos.
- Los modelos más grandes ganan precisión a cambio de velocidad; en CPU conviene el modelo nano.
- Streamlit permite convertir el modelo en una aplicación web pública con pocas líneas de código.

_(Ajusta estas conclusiones con los números que obtuviste en tus experimentos.)_